# 02. Sentinel-2 Multispectral Preprocessing & Spectral Index Derivation
**Project:** Attention U-Net Multispectral Glacier Segmentation Framework  

### Preprocessing Operations:
1. **Physical Surface Reflectance Conversion:** Digital Numbers (DN) scaled to Surface Reflectance:
   $$\rho_\lambda = \frac{\text{DN}}{10,000}$$
2. **Normalized Difference Snow/Ice Index (NDSI):**
   $$\text{NDSI} = \frac{\text{Green (B03)} - \text{SWIR1 (B11)}}{\text{Green (B03)} + \text{SWIR1 (B11)}}$$
3. **Normalized Difference Vegetation Index (NDVI):**
   $$\text{NDVI} = \frac{\text{NIR (B08)} - \text{Red (B04)}}{\text{NIR (B08)} + \text{Red (B04)}}$$
4. **Cloud & Shadow Masking:** Filtering using Sentinel-2 Scene Classification Layer (SCL).
5. **8-Channel Input Tensor Construction:** `[B02, B03, B04, B08, B11, B12, NDSI_norm, NDVI_norm]`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compute_spectral_indices(bands):
    """Computes NDSI and NDVI from 6 Sentinel-2 bands (6, H, W)."""
    b3_green = bands[1].astype(np.float32)
    b4_red = bands[2].astype(np.float32)
    b8_nir = bands[3].astype(np.float32)
    b11_swir1 = bands[4].astype(np.float32)
    eps = 1e-8

    ndsi = (b3_green - b11_swir1) / (b3_green + b11_swir1 + eps)
    ndvi = (b8_nir - b4_red) / (b8_nir + b4_red + eps)

    # Scale indices to [0, 1] range for neural network stability
    ndsi_norm = np.clip((ndsi + 1.0) / 2.0, 0.0, 1.0)
    ndvi_norm = np.clip((ndvi + 1.0) / 2.0, 0.0, 1.0)

    return np.concatenate([bands, ndsi_norm[np.newaxis, ...], ndvi_norm[np.newaxis, ...]], axis=0)

print("[+] Spectral index derivation function compiled.")
